<a href="https://colab.research.google.com/github/pwgacek/ml-project/blob/main/um_gr3_temat6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Time-series Forecasting: Are (Cross-)Attentions Necessary?

### Skład zespołu:

* Paweł Gacek
* Dawid Wołek
* Barbara Wojtarowicz

# DLinear

In [1]:
# Installing conda on colab
%env PYTHONPATH = # /env/python
!wget https://repo.anaconda.com/miniconda/Miniconda3-py38_4.12.0-Linux-x86_64.sh
!chmod +x Miniconda3-py38_4.12.0-Linux-x86_64.sh
!./Miniconda3-py38_4.12.0-Linux-x86_64.sh -b -f -p /usr/local
!conda update conda

import sys
sys.path.append('/usr/local/lib/python3.8/site-packages')

env: PYTHONPATH=# /env/python
--2025-10-19 16:09:49--  https://repo.anaconda.com/miniconda/Miniconda3-py38_4.12.0-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 76120962 (73M) [application/x-sh]
Saving to: ‘Miniconda3-py38_4.12.0-Linux-x86_64.sh’

Miniconda3-py38_4.1 100%[===================>]  72.59M   109MB/s    in 0.7s    

2025-10-19 16:09:50 (109 MB/s) - ‘Miniconda3-py38_4.12.0-Linux-x86_64.sh’ saved [76120962/76120962]

PREFIX=/usr/local
Unpacking payload ...
Solving environment: / - done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - _libgcc_mutex==0.1=main
    - _openmp_mutex==4.5=1_gnu
    - brotlipy==0.7.0=py38h27cfd23_1003
    - ca-certificates==2022.3.29=h06a4308_1
    - certifi==2021.10.8=py38h06a4308_2
    - cffi==

In [2]:
!rm -rdf DLinear-main
!curl -L -o main.zip https://github.com/vivva/DLinear/archive/refs/heads/main.zip
!unzip main.zip
!rm main.zip
!cd DLinear-main && mkdir dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 4745k    0 4745k    0     0  4143k      0 --:--:--  0:00:01 --:--:-- 11.3M
Archive:  main.zip
da9e67442b95af76488b8e4e1806cc3185723dd9
   creating: DLinear-main/
   creating: DLinear-main/FEDformer/
  inflating: DLinear-main/FEDformer/LICENSE  
  inflating: DLinear-main/FEDformer/README.md  
   creating: DLinear-main/FEDformer/data_provider/
  inflating: DLinear-main/FEDformer/data_provider/data_factory.py  
  inflating: DLinear-main/FEDformer/data_provider/data_loader.py  
   creating: DLinear-main/FEDformer/exp/
  inflating: DLinear-main/FEDformer/exp/exp_basic.py  
  inflating: DLinear-main/FEDformer/exp/exp_main.py  
   creating: DLinear-main/FEDformer/layers/
  inflating: DLinear-main/FEDformer/layers/AutoCorrelation.py  
  inflating: DLinear-m

In [3]:
!conda create -n DLinear python=3.6.9
!conda activate DLinear
%pip install -r DLinear-main/requirements.txt

Solving environment: / failed with repodata from current_repodata.json, will retry with next repodata source.
Solving environment: - \ | / done


==> WARNING: A newer version of conda exists. <==
  current version: 4.12.0
  latest version: 25.9.1

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /usr/local/envs/DLinear

  added / updated specs:
    - python=3.6.9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    certifi-2021.5.30          |   py36h06a4308_0         139 KB
    libedit-3.1.20230828       |       h5eee18b_0         179 KB
    libffi-3.2.1               |    hf484d3e_1007          48 KB
    openssl-1.1.1w             |       h7f8727e_0         3.7 MB
    pip-21.2.2                 |   py36h06a4308_0         1.8 MB
    python-3.6.9               |       h265db76_0        30.2 MB
    readline-

In [4]:
%pip install -q gdown
!gdown --id 1Tc7GeVN7DLEl-RAs-JVwG9yFMf--S8dy -O DLinear-main/dataset/weather.csv


     |████████████████████████████████| 106 kB 6.3 MB/s 
/usr/local/lib/python3.8/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1Tc7GeVN7DLEl-RAs-JVwG9yFMf--S8dy
To: /content/DLinear-main/dataset/weather.csv
100% 7.24M/7.24M [00:00<00:00, 19.6MB/s]


In [5]:
!cd DLinear-main && sh scripts/EXP-LongForecasting/DLinear/weather.sh